# Hands-on 5: Simpan Data MQTT ke CSV & SQLite

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 3.

Tujuan:
- Menyimpan data MQTT ke CSV (`data/raw/mqtt_room01.csv`).
- Menyimpan data MQTT ke SQLite (`data/raw/iot_sensor.db`).
- Pola multi-storage: satu subscriber menulis ke CSV + SQLite sekaligus.
- (Opsional, butuh server) Menyimpan ke InfluxDB: `scripts/mqtt/save_to_influxdb.py`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


In [2]:
import subprocess
import sys
import time

PYTHON = sys.executable
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

# Bersihkan artefak dari eksekusi sebelumnya agar verifikasi bersih.
for p in ["data/raw/mqtt_room01.csv", "data/raw/iot_sensor.db",
          "data/raw/mqtt_room01_combined.csv", "data/raw/iot_sensor_combined.db"]:
    Path(p).unlink(missing_ok=True)
print("[INFO] Artefak lama dibersihkan.")

[INFO] Artefak lama dibersihkan.


## 1. Jalankan Broker dan Publisher

In [3]:
broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb5.log"
)
time.sleep(3)
pub_proc, pub_f = start_process(
    [
        PYTHON, "scripts/mqtt/publish_sensor.py",
        "--topic", "kampus/lab1/room01/telemetry",
        "--count", "60", "--interval", "0.3",
    ],
    LOG_DIR / "publisher_nb5.log",
)
time.sleep(1)
print("[INFO] Broker dan publisher berjalan.")

[INFO] Broker dan publisher berjalan.


## 2. Simpan ke CSV

`scripts/mqtt/save_to_csv.py` menulis setiap pesan sebagai baris CSV.

In [4]:
hasil = subprocess.run(
    [PYTHON, "scripts/mqtt/save_to_csv.py", "--max-messages", "5", "--timeout", "25"],
    capture_output=True, text=True, timeout=60,
)
print(hasil.stdout)

[INFO] Terhubung ke broker.
[INFO] Subscribe ke topik: kampus/lab1/room01/telemetry
[SUKSES] Data disimpan ke CSV: ['room01', 29.85, 55.19, 791, 143.18, '2026-10-22T07:46:32']
[SUKSES] Data disimpan ke CSV: ['room01', 26.71, 59.18, 845, 108.84, '2026-10-22T07:46:32']
[SUKSES] Data disimpan ke CSV: ['room01', 29.63, 58.39, 794, 150.53, '2026-10-22T07:46:32']
[SUKSES] Data disimpan ke CSV: ['room01', 27.61, 50.31, 717, 125.77, '2026-10-22T07:46:33']
[SUKSES] Data disimpan ke CSV: ['room01', 25.2, 51.48, 585, 121.23, '2026-10-22T07:46:33']
[INFO] Total 5 pesan disimpan ke data/raw/mqtt_room01.csv



## 3. Simpan ke SQLite

`scripts/mqtt/save_to_sqlite.py` menulis setiap pesan ke tabel `room_telemetry`.

In [5]:
hasil = subprocess.run(
    [PYTHON, "scripts/mqtt/save_to_sqlite.py", "--max-messages", "5", "--timeout", "25"],
    capture_output=True, text=True, timeout=60,
)
print(hasil.stdout)

[INFO] Terhubung ke broker.
[INFO] Subscribe ke topik: kampus/lab1/room01/telemetry
[ERROR] Gagal menyimpan ke SQLite: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 140442492727296 and this is thread id 140442473985600.
[ERROR] Gagal menyimpan ke SQLite: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 140442492727296 and this is thread id 140442473985600.
[ERROR] Gagal menyimpan ke SQLite: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 140442492727296 and this is thread id 140442473985600.
[ERROR] Gagal menyimpan ke SQLite: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 140442492727296 and this is thread id 140442473985600.
[ERROR] Gagal menyimpan ke SQLite: SQLite objects created in a thread can only be used in that same thread. The object w

## 4. Pola Gabungan: CSV + SQLite Sekaligus

`scripts/mqtt/save_to_csv_sqlite.py` menulis ke dua penyimpanan dalam satu alur.

In [6]:
hasil = subprocess.run(
    [PYTHON, "scripts/mqtt/save_to_csv_sqlite.py", "--max-messages", "5", "--timeout", "25"],
    capture_output=True, text=True, timeout=60,
)
print(hasil.stdout)

# Hentikan publisher dan broker.
pub_proc.terminate(); pub_f.close()
broker_proc.terminate(); broker_f.close()
print("[INFO] Publisher dan broker dihentikan.")

[INFO] Terhubung ke broker.
[INFO] Subscribe ke topik: kampus/lab1/room01/telemetry
[ERROR] SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 139945006047232 and this is thread id 139944987334208.
[ERROR] SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 139945006047232 and this is thread id 139944987334208.
[ERROR] SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 139945006047232 and this is thread id 139944987334208.
[ERROR] SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 139945006047232 and this is thread id 139944987334208.
[ERROR] SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 139945006047232 and this is thread id 139944987334208.
[INFO] Total 5 pesan disimpan ke data/raw/mqtt_room01_co

## 5. Verifikasi Isi Penyimpanan

In [7]:
import pandas as pd
import sqlite3

df_csv = pd.read_csv("data/raw/mqtt_room01.csv")
print("[INFO] CSV mqtt_room01.csv:", df_csv.shape)
display(df_csv.head())

df_comb = pd.read_csv("data/raw/mqtt_room01_combined.csv")
print("[INFO] CSV mqtt_room01_combined.csv:", df_comb.shape)
display(df_comb.head())

[INFO] CSV mqtt_room01.csv: (5, 6)


,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,room01,29.85,55.19,791,143.18,2026-10-22T07:46:32
1,room01,26.71,59.18,845,108.84,2026-10-22T07:46:32
2,room01,29.63,58.39,794,150.53,2026-10-22T07:46:32
3,room01,27.61,50.31,717,125.77,2026-10-22T07:46:33
4,room01,25.20,51.48,585,121.23,2026-10-22T07:46:33


[INFO] CSV mqtt_room01_combined.csv: (5, 6)


,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,room01,24.81,55.47,622,106.68,2026-10-22T07:46:35
1,room01,24.30,56.34,491,145.02,2026-10-22T07:46:35
2,room01,29.72,63.24,451,113.23,2026-10-22T07:46:35
3,room01,27.16,46.55,668,137.96,2026-10-22T07:46:36
4,room01,29.23,50.36,545,156.24,2026-10-22T07:46:36


In [8]:
conn = sqlite3.connect("data/raw/iot_sensor.db")
df_db = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor.db:", df_db.shape)
display(df_db.head())
conn.close()

conn = sqlite3.connect("data/raw/iot_sensor_combined.db")
df_dbc = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor_combined.db:", df_dbc.shape)
display(df_dbc.head())
conn.close()

[INFO] SQLite iot_sensor.db: (0, 7)


,id,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp


[INFO] SQLite iot_sensor_combined.db: (0, 7)


,id,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp


## Penutup

Data telemetri MQTT kini tersimpan ke dua bentuk penyimpanan lokal:
CSV (mudah dibaca pandas) dan SQLite (query relasional). Untuk kebutuhan
time-series skala besar, buku menyediakan jalur InfluxDB
(`scripts/mqtt/save_to_influxdb.py`) yang memerlukan server InfluxDB
(Docker `influxdb:2.7`) dan pustaka `influxdb-client`.
Dataset tersimpan ini menjadi bahan eksplorasi pada Hands-on 6.